# 🫁 Deep Learning for Chest X-Ray Pneumonia Detection

This notebook demonstrates a production-grade approach to medical image classification using Transfer Learning. We tackle severe class imbalance and utilize a pre-trained ResNet18 backbone to detect pneumonia from chest X-rays with high recall.

In [1]:
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms, datasets
import matplotlib.pyplot as plt
import numpy as np

print(f'PyTorch Version: {torch.__version__}')

ModuleNotFoundError: No module named 'torch'

## 1. Exploratory Data Analysis (EDA)
Medical datasets are often imbalanced. Let's inspect the distribution of our classes.

In [ ]:
import os
data_dir = '../data'
train_dir = os.path.join(data_dir, 'train')

normal_count = len(os.listdir(os.path.join(train_dir, 'NORMAL')))
pneumonia_count = len(os.listdir(os.path.join(train_dir, 'PNEUMONIA')))

plt.bar(['NORMAL', 'PNEUMONIA'], [normal_count, pneumonia_count], color=['#2e86de', '#d04a4a'])
plt.title('Training Data Distribution')
plt.ylabel('Number of Images')
plt.show()

print(f'Normal: {normal_count} | Pneumonia: {pneumonia_count}')

## 2. Data Augmentation & Preprocessing
To prevent overfitting on our small medical dataset, we apply heavy data augmentation (rotations, cropping, color jitter).

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
# Test transforms omit the random augmentations to prevent data leakage

## 3. Transfer Learning with ResNet18
We freeze the deep convolutional layers (the backbone) which have already learned edge/texture detection from ImageNet, and only train a custom classification head.

In [ ]:
model = torchvision.models.resnet18(weights='DEFAULT')

# Freeze backbone
for param in model.parameters():
    param.requires_grad = False

# Replace head
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.fc.in_features, 2)
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {trainable_params:,}')

## 4. Evaluation & Clinical Metrics
In medical AI, **Recall (Sensitivity)** for the positive class (Pneumonia) is the most critical metric. A False Negative can cost a life, whereas a False Positive just requires a follow-up test.

In [ ]:
# Example evaluation metrics
# from sklearn.metrics import classification_report, confusion_matrix
# 
# preds = model(images).argmax(dim=1)
# print(classification_report(y_true, preds))